# M6: Alpamayo 1.5 — Vision-Language-Action (VLA) Inference

**Stage 7: Model Inference — Alpamayo 1.5 VLA**

| Item | Detail |
|------|--------|
| Model | Alpamayo-1.5-10B (+ hidden Cosmos-Reason2-8B VLM backbone) |
| Instance | ml.p4d.24xlarge / ml.p5.48xlarge, **or** ml.g5.48xlarge (8× A10G) if those are unavailable |
| Input | a pre-saved demo clip (`hf-cache/alpamayo-demo/*.pt`) |
| Output | `users/{profile}/m6/` (predicted trajectory + reasoning) |
| Source | [NVlabs/alpamayo1.5](https://github.com/NVlabs/alpamayo1.5) |

Alpamayo 1.5 is a Vision-Language-Action model: from a short multi-camera clip
plus the ego-vehicle's recent motion, it **reasons about the scene in words**
(a "Chain-of-Causation" explanation) and **predicts the ego trajectory** for the
next few seconds. We compare the prediction to the recorded future with
**minADE** (minimum average displacement error, in metres).

## How this notebook actually runs Alpamayo

The Alpamayo weights are **not** a `pip install`. The real workflow is the
official `NVlabs/alpamayo1.5` repo (package `alpamayo1_5`, Python 3.12), which
we build on the instance NVMe via `scripts/setup_cosmos_env.sh alpamayo`, then
call `scripts/alpamayo_infer.py`. Two things make this fully offline — **you
need no HuggingFace token**:

1. **Model** loads from an S3-restored HuggingFace cache (`HF_HUB_OFFLINE=1`),
   which the admin pre-populated with Alpamayo-1.5-10B **and** its hidden
   Cosmos-Reason2-8B backbone.
2. **Input data** is a demo clip the admin pre-saved with `torch.save` — the
   underlying `PhysicalAI-Autonomous-Vehicles` dataset is gated and cannot be
   read offline, so we ship a ready-made `.pt` instead. This notebook never
   touches HuggingFace for data.

The GPU-check cell picks the placement automatically: a single ≥ 40 GB GPU
(p4d/p5) loads the model on one device; a 24 GB multi-GPU box (g5.48xlarge) uses
`balanced-expert`, which shards the VLM across GPUs but keeps the diffusion
action stack on one GPU. See `docs/ALPAMAYO_M6.md`.

The environment is **ephemeral** (reset on app restart), so the setup cell is
idempotent — re-run it after any restart. First run is ~15–25 min (repo build +
cache restore); afterwards it's near-instant.

## License Notice

**WARNING: Alpamayo 1.5 model weights are under a non-commercial license.**

Commercial use of the model weights requires a separate agreement with NVIDIA.
The inference code in this notebook is licensed under Apache 2.0.

By proceeding, you acknowledge that:
- Model weights are for **research and evaluation purposes only**
- Any commercial deployment requires explicit NVIDIA licensing
- Code contributions remain under Apache 2.0

In [ ]:
# ============================================================
# Config
# ============================================================
import os
import sys
import time
import json
import glob
import subprocess
from pathlib import Path
from datetime import datetime, timezone
import boto3

# HF_TOKEN is NOT needed — the model loads from the admin's offline cache and the
# demo clip is pre-saved. Kept optional purely as an online fallback. Env wins.
HF_TOKEN = os.environ.get("HF_TOKEN", "") or ""
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
PROFILE = os.environ.get("USER_PROFILE", os.environ.get("BLUEPRINT_PROFILE", "default"))
SHARED_BUCKET = os.environ.get("SHARED_BUCKET", f"av30lab-shared-data-{ACCOUNT_ID}")
S3_BUCKET = os.environ.get("USER_BUCKET", f"av30lab-user-workspace-{ACCOUNT_ID}")

# Demo clips: admin pre-saved data dicts (see scripts/alpamayo_save_clip.py).
# Staged under hf-cache/ because the SageMaker execution role can WRITE only
# hf-cache/* on the shared bucket — so the admin uploads the .pt straight from
# the GPU app terminal (datasets/*, model-cache/* are read-only there). Reads
# work from anywhere. No leading slash — composes as f"s3://{BUCKET}/{PREFIX}".
DEMO_PREFIX = "hf-cache/alpamayo-demo/"
DEMO_CLIPS = [
    "030c760c-ae38-49aa-9ad8-f5650a545d26",   # verified (minADE 0.375 m)
    # Uncomment to run the extra staged clips for variety:
    # "<clip_id_2>",
    # "<clip_id_3>",
]
OUTPUT_PREFIX = f"users/{PROFILE}/m6/"

# Local scratch on the instance NVMe (reset on app restart).
NVME = "/mnt/sagemaker-nvme" if os.path.isdir("/mnt/sagemaker-nvme") else "/tmp"
WORK = f"{NVME}/m6_work"
INPUT_DIR = f"{WORK}/clips"
OUTPUT_DIR = f"{WORK}/out"

# Alpamayo env produced by scripts/setup_cosmos_env.sh alpamayo (SEPARATE venv).
COSMOS_WORK = f"{NVME}/cosmos-work"
ALPAMAYO_REPO = f"{COSMOS_WORK}/alpamayo1.5"
ALPAMAYO_ENV = f"{COSMOS_WORK}/alpamayo_env.sh"

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Profile     : {PROFILE}")
print(f"Demo clips  : {DEMO_CLIPS}")
print(f"Input .pt   : s3://{SHARED_BUCKET}/{DEMO_PREFIX}")
print(f"Output      : s3://{S3_BUCKET}/{OUTPUT_PREFIX}")
print(f"NVMe work   : {WORK}")
print(f"HF_TOKEN    : {'set (online fallback)' if HF_TOKEN else 'not set (using admin offline cache — normal)'}")

In [ ]:
# ============================================================
# Pre-flight GPU check + placement strategy
# ============================================================
import torch

assert torch.cuda.is_available(), "CUDA not available — this notebook requires a GPU instance"

gpu_count = torch.cuda.device_count()
print(f"GPUs available: {gpu_count}")
per_device_gb = []
for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    mem_gb = props.total_memory / (1024**3)
    per_device_gb.append(mem_gb)
    print(f"  GPU {i}: {props.name} — {mem_gb:.1f} GB")

# Alpamayo-1.5-10B is ~10.5B params (~21 GB bf16) + VLM-rollout activations.
#  - A single GPU >= 40 GB (p4d A100 / p5 H100): load onto one device (verified).
#  - Otherwise, if the GPUs SUM to enough (e.g. g5.48xlarge = 8x A10G 24 GB =
#    192 GB): use "balanced-expert" — shard the VLM across GPUs but pin the whole
#    action stack (expert/diffusion/action_*) onto cuda:0 so the diffusion KV-cache
#    loop stays on one device. (Plain device_map="auto" splits the expert across
#    GPUs and crashes the diffusion cache — hence balanced-expert.)
max_dev_gb = max(per_device_gb) if per_device_gb else 0
total_gb = sum(per_device_gb)
if max_dev_gb >= 40:
    DEVICE_MAP = ""                 # single-GPU .to("cuda") — the verified path
    print(f"\nLargest single GPU {max_dev_gb:.1f} GB >= 40 GB → single-GPU load (verified path).")
else:
    assert total_gb >= 48 and gpu_count >= 2, (
        f"No single >=40 GB GPU (largest {max_dev_gb:.1f} GB) and only {total_gb:.1f} GB "
        f"total across {gpu_count} GPU(s) — not enough for Alpamayo-1.5-10B. Use p4d/p5, "
        f"or a multi-GPU box (e.g. g5.48xlarge = 8x A10G)."
    )
    DEVICE_MAP = "balanced-expert"  # shard VLM, pin action stack to cuda:0
    print(f"\nNo single >=40 GB GPU; {gpu_count} GPUs sum to {total_gb:.1f} GB "
          f"→ device_map=balanced-expert (g5 fallback path).")

In [ ]:
# ============================================================
# Install the Alpamayo 1.5 environment (idempotent; ~15-25 min first run)
# ============================================================
# Runs scripts/setup_cosmos_env.sh alpamayo: clones NVlabs/alpamayo1.5, builds a
# Python 3.12 uv venv (no flash-attn), restores the S3 HF cache into $HF_HOME,
# and writes alpamayo_env.sh (which flips on HF_HUB_OFFLINE when the cache is
# present). Re-run after any app restart — the NVMe env is ephemeral.

def _find_setup_script():
    for base in [Path.cwd(), Path.cwd().parent, Path.home()]:
        cand = base / "scripts" / "setup_cosmos_env.sh"
        try:
            if cand.exists():
                return str(cand)
        except OSError:
            continue
    return None

setup_script = _find_setup_script()
if setup_script is None:
    # Fall back to the copy staged in S3 (notebook-templates ships scripts/ too).
    local = f"{WORK}/setup_cosmos_env.sh"
    subprocess.run(
        ["aws", "s3", "cp",
         f"s3://{SHARED_BUCKET}/notebook-templates/scripts/setup_cosmos_env.sh", local],
        check=True,
    )
    setup_script = local
print(f"Using setup script: {setup_script}")

env = {**os.environ, "SHARED_BUCKET": SHARED_BUCKET}
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN

proc = subprocess.run(["bash", setup_script, "alpamayo"], env=env, text=True)
if proc.returncode != 0:
    raise RuntimeError(
        "setup_cosmos_env.sh alpamayo failed — scroll up. Common causes: no S3 HF "
        "cache AND no HF_TOKEN, or not enough NVMe space."
    )

assert os.path.exists(ALPAMAYO_ENV), f"Expected {ALPAMAYO_ENV} after setup"
print(f"\nAlpamayo env ready: {ALPAMAYO_ENV}")

In [ ]:
# ============================================================
# Download the pre-saved demo clip(s) + locate the inference script
# ============================================================
# Each demo clip is a `data` dict the admin produced once online with
# scripts/alpamayo_save_clip.py (load_physical_aiavdataset + torch.save, ~100 MB).
# We only torch.load it — physical_ai_av (which can't run offline) is never
# imported here, so no HF token is needed.

def _find_repo_script(name):
    for base in [Path.cwd(), Path.cwd().parent, Path.home()]:
        cand = base / "scripts" / name
        try:
            if cand.exists():
                return str(cand)
        except OSError:
            continue
    return None

infer_script = _find_repo_script("alpamayo_infer.py")
if infer_script is None:
    infer_script = f"{WORK}/alpamayo_infer.py"
    subprocess.run(
        ["aws", "s3", "cp",
         f"s3://{SHARED_BUCKET}/notebook-templates/scripts/alpamayo_infer.py", infer_script],
        check=True,
    )
print(f"Inference script: {infer_script}")

clip_paths = []
for clip in DEMO_CLIPS:
    dst = os.path.join(INPUT_DIR, f"{clip}.pt")
    if not os.path.exists(dst):
        src = f"s3://{SHARED_BUCKET}/{DEMO_PREFIX}{clip}.pt"
        print(f"Downloading {src} ...")
        r = subprocess.run(["aws", "s3", "cp", src, dst, "--quiet"],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"Failed to download demo clip {clip}: {r.stderr}")
    size_mb = os.path.getsize(dst) / 1e6
    assert size_mb > 1.0, f"Demo clip {clip}.pt is only {size_mb:.2f} MB — download incomplete?"
    print(f"  {clip}.pt ({size_mb:.1f} MB)")
    clip_paths.append(dst)

assert clip_paths, "No demo clips downloaded — check DEMO_CLIPS and the S3 prefix."

In [ ]:
# ============================================================
# Run Alpamayo VLA inference (in the a1_5 venv, offline)
# ============================================================
# alpamayo_infer.py loads Alpamayo-1.5-10B once (attn_implementation="sdpa",
# since flash-attn isn't installed) and, per clip, produces the Chain-of-
# Causation reasoning + predicted trajectory + minADE. Runs as a subprocess in
# the Python-3.12 a1_5 venv (the notebook kernel is the SMD 3.12 kernel, a
# different environment). DEVICE_MAP (from the GPU-check cell) is "" for a single
# >=40GB GPU or "balanced-expert" to shard the VLM across smaller GPUs while
# keeping the action stack on cuda:0 (g5/g6 multi-GPU).

def _free_gpus():
    """Reclaim GPU memory before loading the ~21GB model. A prior GPU module
    (M4/M5) OR an interrupted M6 run can leave an orphaned inference process
    pinning VRAM, which makes the Alpamayo load OOM. Kill any leftover
    cosmos/alpamayo inference process and clear the kernel's cache. Safe: no
    such process runs legitimately at this point (cells run in order)."""
    import subprocess as _sp
    for pat in ("scripts/alpamayo_infer.py",
                "cosmos-transfer2.5/examples/inference.py",
                "cosmos-predict2.5/examples/inference.py"):
        _sp.run(["pkill", "-9", "-f", pat], capture_output=True)
    time.sleep(3)
    try:
        import torch as _t
        if _t.cuda.is_available():
            _t.cuda.empty_cache()
    except Exception:
        pass

_free_gpus()
start_time = time.time()

dm_arg = f' --device-map {DEVICE_MAP}' if DEVICE_MAP else ''
cmd = (
    f'source "{ALPAMAYO_ENV}" && cd "{ALPAMAYO_REPO}" && '
    f'python "{infer_script}" --clips {" ".join(clip_paths)} --out "{OUTPUT_DIR}"{dm_arg}'
)
proc = subprocess.run(["bash", "-lc", cmd], text=True, capture_output=True)
print("\n".join((proc.stdout or "").splitlines()[-20:]))
if proc.returncode != 0:
    err_tail = "\n".join((proc.stderr or "").splitlines()[-25:])
    raise RuntimeError(f"Alpamayo inference failed:\n{err_tail}")

with open(os.path.join(OUTPUT_DIR, "metrics.json")) as f:
    metrics = json.load(f)["results"]
print("\nResults:")
for r in metrics:
    print(f"  {r['clip_id']}: minADE = {r['minADE_m']} m  (reasoning {r['cot_chars']} chars)")

In [ ]:
# ============================================================
# Visualize predicted vs. ground-truth trajectory + print reasoning
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

n = len(metrics)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 6), squeeze=False)
for j, r in enumerate(metrics):
    clip_id = r["clip_id"]
    pred = np.load(os.path.join(OUTPUT_DIR, f"{clip_id}_pred.npy"))  # (1, 1, S, T, 3)
    gt = np.load(os.path.join(OUTPUT_DIR, f"{clip_id}_gt.npy"))      # (1, 1, T, 3)
    gt_xy = gt.reshape(-1, gt.shape[-1])[:, :2]
    # Collapse the leading (batch) dims to (S, T, 3); S = trajectory samples.
    pr = pred.reshape(-1, pred.shape[-2], pred.shape[-1])
    ax = axes[0][j]
    for s in range(pr.shape[0]):
        ax.plot(pr[s, :, 0], pr[s, :, 1], "b-o", markersize=3, alpha=0.7,
                label="predicted" if s == 0 else None)
    ax.plot(gt_xy[:, 0], gt_xy[:, 1], "k--s", markersize=3, alpha=0.8, label="ground truth")
    ax.set_title(f"{clip_id[:8]}…  minADE={r['minADE_m']} m")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal", adjustable="datalim")

plt.tight_layout()
viz_path = os.path.join(OUTPUT_DIR, "trajectory_visualization.png")
plt.savefig(viz_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Visualization saved: {viz_path}\n")

# Print the Chain-of-Causation reasoning for each clip.
for r in metrics:
    clip_id = r["clip_id"]
    cot_path = os.path.join(OUTPUT_DIR, f"{clip_id}_cot.txt")
    cot = open(cot_path).read() if os.path.exists(cot_path) else ""
    print("=" * 70)
    print(f"Chain-of-Causation reasoning — {clip_id}")
    print("=" * 70)
    print(cot or "(none)")
    print()

In [ ]:
# ============================================================
# Save outputs to S3
# ============================================================
# Manifest keeps the keys M7 reads (model / modes_run / timestamp / results).
manifest = {
    "module": "M6_Alpamayo_VLA_Inference",
    "profile": PROFILE,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model": "Alpamayo-1.5-10B (alpamayo1_5, sdpa)",
    "license": "non-commercial (model weights), Apache 2.0 (code)",
    "modes_run": ["trajectory_prediction"],
    "clips": DEMO_CLIPS,
    "results": metrics,
}
manifest_path = os.path.join(OUTPUT_DIR, "manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

output_s3_path = f"s3://{S3_BUCKET}/{OUTPUT_PREFIX}"
print(f"Uploading outputs to {output_s3_path} ...")
r = subprocess.run(["aws", "s3", "sync", OUTPUT_DIR, output_s3_path, "--quiet"],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"Upload failed: {r.stderr}")
print(f"Uploaded {len(metrics)} clip result(s) + trajectory visualization + manifest.")

In [ ]:
# ============================================================
# Cost analysis
# ============================================================
# Reads the ACTUAL instance type from SageMaker resource metadata so the estimate
# matches whatever GPU box you launched (g5/g6/p4d/p5) — not a hardcoded p4d rate.
INSTANCE_RATES = {
    "ml.g5.12xlarge": 7.09, "ml.g5.24xlarge": 10.18, "ml.g5.48xlarge": 20.36,
    "ml.g6.12xlarge": 5.53, "ml.g6.24xlarge": 9.84, "ml.g6.48xlarge": 19.69,
    "ml.p4d.24xlarge": 37.69, "ml.p5.48xlarge": 113.14,
}
USD_TO_KRW = 1370

total_elapsed = time.time() - start_time
hours = total_elapsed / 3600

inst = "unknown"
try:
    md = json.loads(Path("/opt/ml/metadata/resource-metadata.json").read_text())
    inst = md.get("InstanceType", "unknown")
except Exception:
    pass
rate = INSTANCE_RATES.get(inst)

print("=" * 50)
print("COST ANALYSIS — M6 Alpamayo VLA Inference")
print("=" * 50)
print(f"Instance:       {inst}")
print(f"Placement:      device_map={DEVICE_MAP or 'single-cuda'}")
print(f"Inference time: {total_elapsed:.0f}s ({hours:.3f} hr)")
if rate is not None:
    cost_usd = hours * rate
    print(f"Rate:           ${rate}/hr")
    print(f"Estimated cost: ${cost_usd:.2f} USD / {cost_usd * USD_TO_KRW:,.0f} KRW")
    print(f"Clips:          {len(metrics)}")
    if metrics:
        print(f"Cost per clip:  ${cost_usd / len(metrics):.4f} USD")
else:
    print(f"Rate:           (unknown instance '{inst}' — not in rate table)")
    print(f"Clips:          {len(metrics)}")
print("Note: setup (repo build + cache restore) is a one-time ~15-25 min cost")
print("      not counted here; it is reset on app restart.")
print("=" * 50)

In [ ]:
# ============================================================
# Output validation + inline preview + next module
# ============================================================
ok = bool(metrics)
print("Output validation:")
for r in metrics:
    clip_id = r["clip_id"]
    pred_f = os.path.join(OUTPUT_DIR, f"{clip_id}_pred.npy")
    exists = os.path.exists(pred_f)
    size_kb = os.path.getsize(pred_f) / 1024 if exists else 0
    valid = exists and size_kb > 0.1 and r["cot_chars"] > 0
    flag = "OK" if valid else "MISSING/EMPTY"
    print(f"    {clip_id}: minADE={r['minADE_m']} m, pred {size_kb:.1f} KB, "
          f"reasoning {r['cot_chars']} chars [{flag}]")
    ok = ok and valid
print(f"  Status: {'PASS' if ok else 'FAIL'}")

# Inline preview of the trajectory plot.
try:
    from IPython.display import Image as IPyImage, display
    if os.path.exists(viz_path):
        display(IPyImage(filename=viz_path))
except Exception as e:
    print(f"(inline preview skipped: {e})")

print(f"\nOutput location: {output_s3_path}")
print("\n" + "=" * 50)
print("NEXT MODULE")
print("=" * 50)
print("M7: AlpaSim Closed-Loop Evaluation")
print("  Evaluates: the SAME Alpamayo-1.5-10B checkpoint, closed-loop")
print("  Reads:     users/{profile}/m6/manifest.json (provenance only —")
print("             the sim runs on a separate GPU EC2, not from these files)")
print("  Reports:   collision_at_fault, collision_rear, offroad,")
print("             dist_to_gt_trajectory")

In [ ]:
"""Mark this module complete on the participant dashboard (best-effort, non-fatal)."""
import sys
from pathlib import Path
for _b in (Path.cwd(), Path.cwd().parent, Path.home()):
    _cand = _b / "scripts" / "av30_progress.py"
    if _cand.exists():
        sys.path.insert(0, str(_b / "scripts"))
        break
try:
    from av30_progress import mark_complete
    mark_complete("m06-alpamayo-vla")
except Exception as _e:
    print(f"[progress] helper unavailable ({_e}); skipping — module still complete.")